# GPT-5.4 phishing inference on `out.jsonl`

This notebook loads `out.jsonl`, prunes each website observation with the same helper logic used in the Gemma4 workflow, sends it to `gpt-5.4`, and writes predictions to `gpt54_out_predictions.jsonl`.

Set `OPENAI_API_KEY` in your environment before running it.

In [8]:
import json
import os
import random
import re
import time
from pathlib import Path
from typing import Any

from openai import OpenAI
from tqdm.auto import tqdm

DATA_PATH = Path("out.jsonl")
OUTPUT_PATH = Path("gpt54_out_predictions.jsonl")
MODEL = "gpt-5.4-mini"
SAMPLE_SIZE = 20000  # Total records to score. Must be even, or None to use the full balanced set.
RANDOM_SEED = 67
SLEEP_BETWEEN_REQUESTS_SEC = 0.0
REASONING_EFFORT = "none"
VERBOSITY = "low"

SYSTEM_PROMPT = (
    "You are a phishing website classifier. "
    "Return only valid JSON. Do not wrap it in Markdown. "
    "Use exactly this top-level structure: "
    '{"verdict":"phishing|benign","confidence_level":"low|medium|high",'
    '"evidence":[{"id":"feature.id","direction":"suspicious|benign|neutral",'
    '"severity":"low|medium|high","statement":"short human-readable reason"}]}. '
    "Allowed verdict values are phishing, benign. "
    "Allowed confidence_level values are low, medium, high. "
    "Evidence items must use keys id, direction, severity, value, and statement. "
    "Use only observable artifacts from the provided website observation. "
    "Do not mention dataset source, training labels, collection names, or hidden metadata."
    "Feature IDs should be specific and descriptive, e.g. 'url.path_length' etc."
    "Do not give any brand specific evidence. Given evidence should be widely applicable."
    "Content classified as phishing can also have benign indicators, and content classified as benign can have suspicious indicators. They may also not."
)

def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in {'"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)


load_env_file(Path(".env"))

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH.resolve()}")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the environment or in a local .env file before running this notebook.")

client = OpenAI()

In [9]:
def compact_text(value: Any, max_chars: int) -> str:
    text = re.sub(r"\s+", " ", str(value or "")).strip()
    return text[:max_chars].rstrip()


def redact_large_value(value: Any, max_chars: int = 240) -> Any:
    if isinstance(value, str):
        if value.startswith("data:"):
            return value[:48] + "<data_uri_truncated>"
        return compact_text(value, max_chars)
    if isinstance(value, list):
        return [redact_large_value(item, max_chars) for item in value[:20]]
    if isinstance(value, dict):
        return {key: redact_large_value(child, max_chars) for key, child in value.items()}
    return value


def collection_items(value: Any) -> list[Any]:
    if isinstance(value, dict):
        items = value.get("items")
        return items if isinstance(items, list) else []
    if isinstance(value, list):
        return value
    return []


def collection_total(value: Any) -> int:
    if isinstance(value, dict):
        total = value.get("total_observed")
        if isinstance(total, int):
            return total
        return len(collection_items(value))
    if isinstance(value, list):
        return len(value)
    return 0


def prune_input(page_input: dict[str, Any]) -> dict[str, Any]:
    """Keep model-visible website observation compact and source/label free."""
    page_input = page_input or {}
    resources = page_input.get("resources") or {}
    forms = page_input.get("forms")
    anchors = page_input.get("anchors")
    iframes = page_input.get("iframes")
    pruned = {
        "url": compact_text(page_input.get("url"), 1000),
        "final_url": compact_text(page_input.get("final_url"), 1000),
        "redirects": redact_large_value((page_input.get("redirects") or [])[:10]),
        "title": compact_text(page_input.get("title"), 300),
        "meta": redact_large_value((page_input.get("meta") or [])[:20], 240),
        "visible_text": compact_text(page_input.get("visible_text"), 3500),
        "forms": {
            "total_observed": collection_total(forms),
            "items": redact_large_value(collection_items(forms)[:20], 240),
        },
        "anchors": {
            "total_observed": collection_total(anchors),
            "items": redact_large_value(collection_items(anchors)[:30], 200),
        },
        "iframes": {
            "total_observed": collection_total(iframes),
            "items": redact_large_value(collection_items(iframes)[:10], 200),
        },
        "resources": {
            "favicon_hrefs": redact_large_value((resources.get("favicon_hrefs") or [])[:10], 200),
            "script_src_sample": redact_large_value((resources.get("script_src_sample") or [])[:20], 200),
            "stylesheet_href_sample": redact_large_value((resources.get("stylesheet_href_sample") or [])[:20], 200),
            "image_src_sample": redact_large_value((resources.get("image_src_sample") or [])[:10], 120),
        },
    }
    return {key: value for key, value in pruned.items() if value not in ("", None)}


def make_user_payload(record: dict[str, Any]) -> str:
    payload = {
        "task": "Classify the website as phishing, benign, or uncertain and return strict JSON.",
        "website_observation": prune_input(record.get("input") or {}),
    }
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


def extract_first_json_object(text: str) -> dict[str, Any] | None:
    if not text:
        return None
    text = text.strip()
    try:
        parsed = json.loads(text)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    in_string = False
    escape = False
    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    try:
                        parsed = json.loads(text[start : index + 1])
                        return parsed if isinstance(parsed, dict) else None
                    except json.JSONDecodeError:
                        return None
    return None


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_number} of {path}") from exc
    return rows


def balanced_random_sample(records: list[dict[str, Any]], sample_size: int | None, seed: int) -> list[dict[str, Any]]:
    grouped = {
        "phishing": [record for record in records if record.get("label") == "phishing"],
        "benign": [record for record in records if record.get("label") == "benign"],
    }
    if not grouped["phishing"] or not grouped["benign"]:
        raise ValueError("Balanced sampling requires both phishing and benign records.")
    if sample_size is None:
        per_label = min(len(grouped["phishing"]), len(grouped["benign"]))
    else:
        if sample_size <= 0 or sample_size % 2 != 0:
            raise ValueError("SAMPLE_SIZE must be a positive even integer, or None.")
        per_label = sample_size // 2
    if len(grouped["phishing"]) < per_label or len(grouped["benign"]) < per_label:
        raise ValueError(
            f"Not enough records for balanced sampling: need {per_label} per label, "
            f"have phishing={len(grouped['phishing'])}, benign={len(grouped['benign'])}."
        )
    rng = random.Random(seed)
    sampled = rng.sample(grouped["phishing"], per_label) + rng.sample(grouped["benign"], per_label)
    rng.shuffle(sampled)
    return sampled


In [10]:
all_records = load_jsonl(DATA_PATH)
records = balanced_random_sample(all_records, SAMPLE_SIZE, RANDOM_SEED)

label_counts = {}
for label in ("phishing", "benign"):
    label_counts[label] = sum(1 for record in records if record.get("label") == label)

print(f"Loaded {len(records)} randomly sampled balanced records from {DATA_PATH}")
print({"sample_size": len(records), "seed": RANDOM_SEED, "label_counts": label_counts})
sample_payload = json.loads(make_user_payload(records[0]))
print(json.dumps(sample_payload, ensure_ascii=False, indent=2)[:2500])

Loaded 20000 randomly sampled balanced records from out.jsonl
{'sample_size': 20000, 'seed': 67, 'label_counts': {'phishing': 10000, 'benign': 10000}}
{
  "task": "Classify the website as phishing, benign, or uncertain and return strict JSON.",
  "website_observation": {
    "url": "https://jamanetwork.com/signinshibboleth?returnUrl=https%3a%2f%2fjamanetwork.com%2f",
    "final_url": "https://jamanetwork.com/signinshibboleth?returnUrl=https%3a%2f%2fjamanetwork.com%2f",
    "redirects": [],
    "title": "Sign In via Shibboleth",
    "meta": [
      {
        "http_equiv": "Content-Type",
        "content": "text/html; charset=utf-8"
      },
      {
        "name": "viewport",
        "content": "width=device-width, initial-scale=1"
      },
      {
        "name": "format-detection",
        "content": "telephone=no"
      },
      {
        "http_equiv": "X-UA-Compatible",
        "content": "IE=Edge"
      },
      {
        "name": "ROBOTS",
        "content": "NOINDEX, NOFOLLOW"
  

In [11]:
results = []
written_count = 0

progress = tqdm(records, total=len(records), desc="Scoring records", unit="record")

with OUTPUT_PATH.open("w", encoding="utf-8") as handle:
    for index, record in enumerate(progress, start=1):
        user_payload = make_user_payload(record)
        row = {
            "id": record.get("id"),
            "label": record.get("label"),
            "model": MODEL,
            "request": json.loads(user_payload),
        }
        try:
            response = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_payload},
                ],
                reasoning={"effort": REASONING_EFFORT},
                text={"verbosity": VERBOSITY},
                store=False,
            )
            output_raw = (response.output_text or "").strip()
            row["output_raw"] = output_raw
            row["output_json"] = extract_first_json_object(output_raw)
            verdict = (row["output_json"] or {}).get("verdict", "unparseable")
            progress.set_postfix_str(f"{row['id']} -> {verdict}")
        except Exception as exc:
            row["error"] = str(exc)
            progress.set_postfix_str(f"{row['id']} -> ERROR")
        if isinstance(row.get("output_json"), dict):
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
            written_count += 1
        results.append(row)
        if SLEEP_BETWEEN_REQUESTS_SEC:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

print(f"Wrote {written_count} rows to {OUTPUT_PATH.resolve()}")

Scoring records:   0%|          | 0/20000 [00:00<?, ?record/s]

KeyboardInterrupt: 

In [ ]:
parsed_count = sum(1 for row in results if isinstance(row.get("output_json"), dict))
error_count = sum(1 for row in results if row.get("error"))
print({
    "total": len(results),
    "parsed_json": parsed_count,
    "errors": error_count,
    "output_path": str(OUTPUT_PATH.resolve()),
})

results[:2]

{'total': 100, 'parsed_json': 100, 'errors': 0, 'output_path': '/home/ege/Documents/Projects/XAI_Analyzer/gpt54_out_predictions.jsonl'}


[{'id': '6932476c064a766321b54b82',
  'label': 'phishing',
  'model': 'gpt-5.4-mini',
  'request': {'task': 'Classify the website as phishing, benign, or uncertain and return strict JSON.',
   'website_observation': {'url': 'http://pub-3308d044822d418eb107c4304d862d79.r2.dev/auth_type.html?folder=wg4cs80adh',
    'final_url': 'https://pub-3308d044822d418eb107c4304d862d79.r2.dev/auth_type.html?folder=wg4cs80adh',
    'redirects': [{'status_code': 301,
      'url': 'http://pub-3308d044822d418eb107c4304d862d79.r2.dev/auth_type.html?folder=wg4cs80adh'}],
    'title': 'Webmail Portal Access',
    'meta': [],
    'visible_text': "Sign in to continue Enter your correct password to avoid deactivation Invalid credentials. email/password is incorrect That account doesn't exist. Enter a different account Remember me Sign in",
    'forms': {'total_observed': 1,
     'items': [{'method': 'post',
       'action': '#',
       'text': "Sign in to continue Enter your correct password to avoid deactivat

In [ ]:
from collections import Counter


def get_analysis_results() -> list[dict[str, Any]]:
    data = globals().get("results")
    if isinstance(data, list) and data:
        return data
    if not OUTPUT_PATH.exists():
        raise FileNotFoundError(f"No in-memory results and no saved file at {OUTPUT_PATH.resolve()}")
    data = load_jsonl(OUTPUT_PATH)
    print(f"Loaded {len(data)} saved results from {OUTPUT_PATH}")
    return data


def build_label_by_id() -> dict[str, str]:
    source_records = globals().get("all_records")
    if not isinstance(source_records, list) or not source_records:
        source_records = load_jsonl(DATA_PATH)
    return {
        str(record.get("id")): str(record.get("label"))
        for record in source_records
        if record.get("id") is not None and record.get("label") is not None
    }


def average(values: list[int | float]) -> float:
    return round(sum(values) / len(values), 3) if values else 0.0


def normalize_analysis_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    label_by_id = build_label_by_id()
    normalized = []
    for row in rows:
        row_id = str(row.get("id")) if row.get("id") is not None else None
        output_json = row.get("output_json") if isinstance(row.get("output_json"), dict) else None
        evidence = output_json.get("evidence") if output_json else []
        if not isinstance(evidence, list):
            evidence = []
        evidence = [item for item in evidence if isinstance(item, dict)]
        feature_ids = [str(item.get("id")) for item in evidence if item.get("id")]
        normalized.append(
            {
                "id": row_id,
                "label": row.get("label") or label_by_id.get(row_id),
                "predicted_verdict": (output_json or {}).get("verdict", "unparseable"),
                "confidence_level": (output_json or {}).get("confidence_level"),
                "evidence": evidence,
                "evidence_count": len(evidence),
                "feature_ids": feature_ids,
                "error": row.get("error"),
                "has_parseable_json": output_json is not None,
            }
        )
    return normalized


analysis_rows = normalize_analysis_rows(get_analysis_results())
print(f"Prepared {len(analysis_rows)} rows for analysis")

Prepared 100 rows for analysis


In [ ]:
distinct_feature_ids = sorted({feature_id for row in analysis_rows for feature_id in row["feature_ids"]})
verdict_counts = Counter(row["predicted_verdict"] for row in analysis_rows)
confidence_counts = Counter(row["confidence_level"] or "missing" for row in analysis_rows)
label_counts = Counter(row["label"] or "missing" for row in analysis_rows)

summary = {
    "rows": len(analysis_rows),
    "label_counts": dict(label_counts),
    "predicted_verdict_counts": dict(verdict_counts),
    "confidence_level_counts": dict(confidence_counts),
    "parseable_json_rate": round(sum(1 for row in analysis_rows if row["has_parseable_json"]) / len(analysis_rows), 3),
    "error_rate": round(sum(1 for row in analysis_rows if row["error"]) / len(analysis_rows), 3),
    "avg_feature_count_all_rows": average([row["evidence_count"] for row in analysis_rows]),
    "avg_feature_count_parsed_rows": average([row["evidence_count"] for row in analysis_rows if row["has_parseable_json"]]),
    "avg_feature_count_by_label": {
        label: average([row["evidence_count"] for row in analysis_rows if row["label"] == label])
        for label in sorted(label_counts)
    },
    "avg_feature_count_by_predicted_verdict": {
        verdict: average([row["evidence_count"] for row in analysis_rows if row["predicted_verdict"] == verdict])
        for verdict in sorted(verdict_counts)
    },
    "distinct_feature_id_count": len(distinct_feature_ids),
}

print(json.dumps(summary, ensure_ascii=False, indent=2))
distinct_feature_ids

{
  "rows": 100,
  "label_counts": {
    "phishing": 50,
    "benign": 50
  },
  "predicted_verdict_counts": {
    "phishing": 46,
    "benign": 50,
    "uncertain": 4
  },
  "confidence_level_counts": {
    "high": 73,
    "medium": 26,
    "low": 1
  },
  "parseable_json_rate": 1.0,
  "error_rate": 0.0,
  "avg_feature_count_all_rows": 5.51,
  "avg_feature_count_parsed_rows": 5.51,
  "avg_feature_count_by_label": {
    "benign": 5.14,
    "phishing": 5.88
  },
  "avg_feature_count_by_predicted_verdict": {
    "benign": 5.14,
    "phishing": 5.957,
    "uncertain": 5.0
  },
  "distinct_feature_id_count": 381
}


['account_action_links',
 'anchor.external_shortlink',
 'anchor.external_unrelated_link',
 'anchor.login_signup_links',
 'anchor.password_reset_external',
 'anchors.account_links_present',
 'anchors.all_to_same_target',
 'anchors.contact_navigation',
 'anchors.content_links',
 'anchors.count',
 'anchors.destination_same_domain',
 'anchors.external_brand_link',
 'anchors.external_microsoft_help_link',
 'anchors.external_misleading_link',
 'anchors.external_nginx_links',
 'anchors.external_official_services',
 'anchors.external_random_link',
 'anchors.internal_navigation',
 'anchors.javascript_links',
 'anchors.language_menu',
 'anchors.language_switcher',
 'anchors.login_links',
 'anchors.official_links',
 'anchors.official_support_links',
 'anchors.platform_link',
 'anchors.terms_privacy_links',
 'anchors.total_observed',
 'brand.impersonation.title',
 'brand.impersonation.visible_text',
 'branding.impersonation',
 'branding.official_branding_copy',
 'commerce_functionality',
 'contact

In [ ]:
feature_id_counts = Counter(feature_id for row in analysis_rows for feature_id in row["feature_ids"])
direction_counts = Counter(
    str(item.get("direction") or "missing")
    for row in analysis_rows
    for item in row["evidence"]
)
severity_counts = Counter(
    str(item.get("severity") or "missing")
    for row in analysis_rows
    for item in row["evidence"]
)

feature_summary = {
    "top_feature_ids": feature_id_counts.most_common(25),
    "direction_counts": dict(direction_counts),
    "severity_counts": dict(severity_counts),
}

print(json.dumps(feature_summary, ensure_ascii=False, indent=2))

{
  "top_feature_ids": [
    [
      "html.forms_count",
      22
    ],
    [
      "url.domain",
      20
    ],
    [
      "title.brand_impersonation",
      17
    ],
    [
      "forms.total_observed",
      13
    ],
    [
      "page.title",
      9
    ],
    [
      "url.domain_brand_match",
      7
    ],
    [
      "visible_text.brand_impersonation",
      7
    ],
    [
      "url.domain_mismatch",
      6
    ],
    [
      "url.domain_brand_mismatch",
      5
    ],
    [
      "url.lookalike_domain",
      5
    ],
    [
      "meta.og_url_mismatch",
      4
    ],
    [
      "title.brand_mismatch",
      4
    ],
    [
      "page.title_branding",
      4
    ],
    [
      "visible_text.login_prompt",
      4
    ],
    [
      "url.brand_mismatch",
      4
    ],
    [
      "forms.search_only",
      4
    ],
    [
      "url.domain_match",
      3
    ],
    [
      "content.brand_consistency",
      3
    ],
    [
      "url.domain_branding",
      3
    ],
    

In [ ]:
verdict_order = ["phishing", "benign", "uncertain", "unparseable"]
confusion = {
    label: {
        verdict: sum(1 for row in analysis_rows if row["label"] == label and row["predicted_verdict"] == verdict)
        for verdict in verdict_order
    }
    for label in sorted({row["label"] for row in analysis_rows if row["label"]})
}

misclassified_examples = [
    {
        "id": row["id"],
        "label": row["label"],
        "predicted_verdict": row["predicted_verdict"],
        "confidence_level": row["confidence_level"],
        "feature_ids": row["feature_ids"],
    }
    for row in analysis_rows
    if row["label"] in {"phishing", "benign"} and row["predicted_verdict"] not in {row["label"], "unparseable"}
]

print(json.dumps({"confusion": confusion, "misclassified_count": len(misclassified_examples)}, ensure_ascii=False, indent=2))
misclassified_examples[:10]

{
  "confusion": {
    "benign": {
      "phishing": 1,
      "benign": 47,
      "uncertain": 2,
      "unparseable": 0
    },
    "phishing": {
      "phishing": 45,
      "benign": 3,
      "uncertain": 2,
      "unparseable": 0
    }
  },
  "misclassified_count": 8
}


[{'id': '6927d21f5c210b46a4300afc',
  'label': 'phishing',
  'predicted_verdict': 'benign',
  'confidence_level': 'medium',
  'feature_ids': ['content.brand_consistency',
   'html.forms_count',
   'url.domain',
   'content.purpose',
   'links.official_domains']},
 {'id': '6928ff125c210b46a4300f49',
  'label': 'phishing',
  'predicted_verdict': 'benign',
  'confidence_level': 'medium',
  'feature_ids': ['url.domain_branding',
   'page.title_generic',
   'meta.description_generic',
   'forms.login_like',
   'iframe.editorbar',
   'anchors.platform_link']},
 {'id': '69ce4d7e261193eaca6895df',
  'label': 'benign',
  'predicted_verdict': 'uncertain',
  'confidence_level': 'medium',
  'feature_ids': ['url.domain_reputation',
   'content.topic_consistency',
   'html.forms_count',
   'content_template_mismatch',
   'resources.external_scripts']},
 {'id': '693b207976ba939058808f04',
  'label': 'phishing',
  'predicted_verdict': 'benign',
  'confidence_level': 'medium',
  'feature_ids': ['url.do